# Análisis Exploratorio y Partición de Datos

Notebook de la etapa **2** del pipeline de detección de cyberbullying en texto.
Su rol es servir de **eslabón** entre la extracción de características (01) y el
modelado (03): aquí se describe la estructura del dataset preprocesado, se
evidencian propiedades relevantes (desbalance de clases, textos duplicados,
longitud de los mensajes) y se define la partición entrenamiento/test que luego
se utiliza en el modelado. Este análisis alimenta la sección metodológica de la
tesis.

> El dataset de entrada es `../../data/cyberbullying_preprocessed.csv`, generado por
> la etapa 00 (columnas `text`, `label`, `text_preprocessed`). Este notebook está
> **sin ejecutar**; los resultados se completarán al correrlo en el run final.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Cargar el dataset preprocesado
df = pd.read_csv('../../data/cyberbullying_preprocessed.csv')

In [ ]:
# Estructura y calidad del dataset
print('Forma del dataset:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)

# Textos vacíos tras el preprocesado (cadenas vacías o nulas)
empty_texts = (df['text_preprocessed'].isna() | (df['text_preprocessed'].str.strip() == '')).sum()
print(f'\nFilas con texto vacío o nulo en text_preprocessed: {empty_texts}')

# Valores nulos generales
print('\nValores nulos por columna:')
print(df.isna().sum())

In [ ]:
# Distribución de clases: conteo y proporción (porcentaje)
# Evidencia si existe desbalance entre las clases de la variable objetivo.
class_counts = df['label'].value_counts()
class_proportions = df['label'].value_counts(normalize=True) * 100

class_dist_df = pd.DataFrame({
    'Conteo': class_counts,
    'Proporción (%)': class_proportions.round(2)
})
class_dist_df

In [ ]:
# Longitud de los textos por clase (en caracteres y en palabras)
df['n_chars'] = df['text_preprocessed'].str.len()
df['n_words'] = df['text_preprocessed'].str.split().str.len()

print('Longitud en caracteres por clase:')
print(df.groupby('label')['n_chars'].describe())

print('\nLongitud en palabras por clase:')
print(df.groupby('label')['n_words'].describe())

# Boxplot de la longitud (en palabras) por clase
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='label', y='n_words')
plt.title('Longitud en palabras por clase')
plt.show()

In [ ]:
from collections import Counter

# Top N palabras más frecuentes por clase (conteo simple tras el split)
TOP_N = 20

for label_value in sorted(df['label'].unique()):
    word_counter = Counter()
    for doc in df[df['label'] == label_value]['text_preprocessed']:
        word_counter.update(doc.split())
    top_words = pd.DataFrame(word_counter.most_common(TOP_N), columns=['word', 'count'])
    print(f'\nTop {TOP_N} palabras — clase {label_value}:')
    print(top_words.to_string(index=False))

In [ ]:
# Detección de duplicados en el texto preprocesado.
# Los tweets duplicados (mensajes repetidos o retuiteados) inflan artificialmente
# la evaluación: si un mismo texto aparece en entrenamiento y test, el modelo lo
# "recuerda" y las métricas sobreestiman el rendimiento real.
is_duplicate = df['text_preprocessed'].duplicated(keep=False)
n_duplicated_rows = is_duplicate.sum()
print(f'Filas que comparten texto con otra(s): {n_duplicated_rows} '
      f'({n_duplicated_rows / len(df) * 100:.2f}% del dataset)')

# Muestras de texto duplicado
df[is_duplicate].head()

In [ ]:
# Partición entrenamiento/test con la MISMA configuración que la etapa 03:
# test_size=0.2, random_state=1 y stratify=y, de modo que los splits coincidan
# exactamente entre este notebook y el de modelado.
X = df['text_preprocessed']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

print(f'Tamaño de entrenamiento: {X_train.shape[0]}')
print(f'Tamaño de test: {X_test.shape[0]}')

# Verificar que el balance de clases se mantiene en cada partición
print('\nProporción de la clase 1 (bullying) en cada partición:')
print(f'  Dataset completo: {y.mean():.4f}')
print(f'  Entrenamiento:    {y_train.mean():.4f}')
print(f'  Test:             {y_test.mean():.4f}')

## Interpretación de los resultados del EDA

El análisis exploratorio revela propiedades del dataset que condicionan la
estrategia de modelado:

- **Desbalance de clases**: la distribución de `label` no es equilibrada (la
  clase de bullying concentra aproximadamente el 61 % de los mensajes frente al
  39 % de la clase negativa). En estas condiciones, la **accuracy** es una
  métrica engañosa: un clasificador que predice siempre la clase mayoritaria
  obtendría una accuracy alta sin detectar correctamente el caso de interés.
- **Duplicados**: los textos duplicados (frecuentes en tweets) pueden inflar la
  evaluación si un mismo mensaje aparece simultáneamente en entrenamiento y test.
- **Longitud**: comparar la distribución de longitud por clase aporta contexto
  interpretativo: permite observar si los mensajes de bullying difieren
  sistemáticamente en extensión respecto de los no bullying.

**Consecuencia metodológica**: por el desbalance, la **F1-score de la clase de
bullying** se adopta como métrica principal de selección y comparación de
modelos, en lugar de la accuracy. Esta decisión se refleja en la etapa 03, donde
los modelos se ordenan y comparan por F1 y se reporta además precisión y recall.